In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from scipy.stats import zscore
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:


df_mon = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv")
df_tue = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv")
df_wed = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv")
df_thu_morning = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv")
df_thu_afternoon = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv")
df_fri_morning = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv")
df_fri_afternoon_ddos = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
df_fri_afternoon_pscan = pd.read_csv("../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")

df_all_data = pd.concat([df_mon, df_tue, df_wed, df_thu_morning, df_thu_afternoon, df_fri_morning, df_fri_afternoon_ddos, df_fri_afternoon_pscan], ignore_index=True)
df_all_data.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,49188,4,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,49188,1,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,49486,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [6]:
df_all_data.shape

(2830743, 79)

In [7]:
def load_cicids2017(dataset_path):
    """
    Loads and concatenates all CIC-IDS2017 CSV files.
    Handles common issues like missing columns.
    """
    all_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith('.csv')]
    print(f"Found {len(all_files)} CSV files.")
    df_list = []
    print(f"Loading CIC-IDS2017 from: {dataset_path}")
    for i, file in enumerate(all_files):
        try:
            print(f"  Loading {os.path.basename(file)} ({i+1}/{len(all_files)})")
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error loading {file}: {e}")
            continue
    if not df_list:
        raise ValueError("No CIC-IDS2017 files loaded. Check your path and file types.")

    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Clean column names: remove leading/trailing spaces and convert to snake_case
    combined_df.columns = combined_df.columns.str.strip().str.replace(' ', '_').str.replace('/', '_').str.lower()
    
    print(f"CIC-IDS2017 loaded. Shape: {combined_df.shape}")
    return combined_df

In [8]:
all_data_df = load_cicids2017('../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/')
all_data_df

Found 8 CSV files.
Loading CIC-IDS2017 from: ../data/CIC-IDS2017/MachineLearningCSV/MachineLearningCSV/MachineLearningCVE/
  Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv (1/8)
  Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv (2/8)
  Loading Friday-WorkingHours-Morning.pcap_ISCX.csv (3/8)
  Loading Monday-WorkingHours.pcap_ISCX.csv (4/8)
  Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv (5/8)
  Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv (6/8)
  Loading Tuesday-WorkingHours.pcap_ISCX.csv (7/8)
  Loading Wednesday-workingHours.pcap_ISCX.csv (8/8)
CIC-IDS2017 loaded. Shape: (2830743, 79)


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2830738,53,32215,4,2,112,152,28,28,28.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830739,53,324,2,2,84,362,42,42,42.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830740,58030,82,2,1,31,6,31,0,15.5,21.92031,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2830741,53,1048635,6,2,192,256,32,32,32.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
# Check if there is a date/time column in all_data_df
print(all_data_df.columns)

# If there is a column representing date/time, for example 'timestamp' or similar, use it to sort:
# Replace 'timestamp' with the actual column name if different
if 'timestamp' in all_data_df.columns:
    all_data_df = all_data_df.sort_values(by='timestamp')
else:
    print("No date/time column found in all_data_df to sort by.")

Index(['destination_port', 'flow_duration', 'total_fwd_packets',
       'total_backward_packets', 'total_length_of_fwd_packets',
       'total_length_of_bwd_packets', 'fwd_packet_length_max',
       'fwd_packet_length_min', 'fwd_packet_length_mean',
       'fwd_packet_length_std', 'bwd_packet_length_max',
       'bwd_packet_length_min', 'bwd_packet_length_mean',
       'bwd_packet_length_std', 'flow_bytes_s', 'flow_packets_s',
       'flow_iat_mean', 'flow_iat_std', 'flow_iat_max', 'flow_iat_min',
       'fwd_iat_total', 'fwd_iat_mean', 'fwd_iat_std', 'fwd_iat_max',
       'fwd_iat_min', 'bwd_iat_total', 'bwd_iat_mean', 'bwd_iat_std',
       'bwd_iat_max', 'bwd_iat_min', 'fwd_psh_flags', 'bwd_psh_flags',
       'fwd_urg_flags', 'bwd_urg_flags', 'fwd_header_length',
       'bwd_header_length', 'fwd_packets_s', 'bwd_packets_s',
       'min_packet_length', 'max_packet_length', 'packet_length_mean',
       'packet_length_std', 'packet_length_variance', 'fin_flag_count',
       'syn_flag_co

In [10]:
def handle_missing_values(df):
    """
    Handles missing values using median for numerical and mode for categorical.
    """
    print("Handling missing values...")
    # Replace infinite values with NaN first
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    for col in df.columns:
        if df[col].isnull().any():
            if df[col].dtype in ['int66', 'float64']:
                # For numerical columns, use median imputation
                median_val = df[col].median()
                df[col].fillna(median_val, inplace=True)
                print(f"  Imputed numerical column '{col}' with median: {median_val}")
            else:
                # For categorical columns, use mode imputation
                mode_val = df[col].mode()[0]
                df[col].fillna(mode_val, inplace=True)
                print(f"  Imputed categorical column '{col}' with mode: {mode_val}")
    print("Missing value handling complete.")
    return df

In [11]:
imputed_df = handle_missing_values(all_data_df)
imputed_df.head()

Handling missing values...


C:\Users\vicky\AppData\Local\Temp\ipykernel_33352\2749718181.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)
C:\Users\vicky\AppData\Local\Temp\ipykernel_33352\2749718181.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, 

  Imputed numerical column 'flow_bytes_s' with median: 4586.600756
  Imputed numerical column 'flow_packets_s' with median: 109.4759992228
Missing value handling complete.


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [16]:
def detect_and_handle_outliers_zscore(df, threshold=3):
    """
    Detects and handles outliers using the Z-score method.
    Outliers are capped to the threshold (e.g., 3 standard deviations from mean).
    Applies only to numerical columns.
    """
    print(f"Detecting and handling outliers using Z-score (threshold={threshold})...")
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

    for col in numerical_cols:
        # Calculate Z-scores
        z_scores = np.abs(zscore(df[col]))

        # Identify outliers
        outlier_indices = df.index[z_scores > threshold].tolist()
        if outlier_indices:
            # print(f"  Found {len(outlier_indices)} outliers in column '{col}'.")
            # Capping: replace outliers with the value at the threshold
            mean_val = df[col].mean()
            std_dev = df[col].std()
            
            # Cap values beyond +threshold*std_dev to +threshold*std_dev
            df.loc[z_scores > threshold, col] = np.sign(df[col]) * (mean_val + threshold * std_dev) # Cap to 3 std dev from mean
            
            # Alternative: Replace with median of non-outliers
            # non_outliers = df[col][z_scores <= threshold]
            # if not non_outliers.empty:
            #     df.loc[z_scores > threshold, col] = non_outliers.median()
            # else:
            #     df.loc[z_scores > threshold, col] = df[col].median() # Fallback if no non-outliers

    print("Outlier handling complete.")
    return df

In [17]:
outlier_processed_df = detect_and_handle_outliers_zscore(imputed_df)
outlier_processed_df.head()

Detecting and handling outliers using Z-score (threshold=3)...
Outlier handling complete.


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,55054.0,109.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,55055.0,52.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,46236.0,34.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,54863.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


In [18]:
def encode_categorical_features(df, target_column=None):
    """
    Encodes categorical features using One-Hot Encoding for features
    and Label Encoding for the target column.
    """
    print("Encoding categorical features...")
    categorical_cols = df.select_dtypes(include='object').columns.tolist()

    if target_column and target_column in categorical_cols:
        # Label Encode the target column
        le = LabelEncoder()
        df[target_column] = le.fit_transform(df[target_column])
        print(f"  Label encoded target column: '{target_column}'")
        # Remove target from categorical_cols for one-hot encoding
        categorical_cols.remove(target_column)
        
        # Store the label encoder for inverse transformation if needed later
        global target_label_encoder
        target_label_encoder = le

    if categorical_cols:
        # One-Hot Encode other categorical features
        print(f"  One-hot encoding features: {categorical_cols}")
        df = pd.get_dummies(df, columns=categorical_cols, drop_first=True) # drop_first to avoid multicollinearity
    else:
        print("  No other categorical features to encode.")
    
    print("Categorical feature encoding complete.")
    return df

In [19]:
encoded_df = encode_categorical_features(outlier_processed_df, target_column='label')  # Replace 'label' with your actual target column name
encoded_df.head()

Encoding categorical features...
  Label encoded target column: 'label'
  No other categorical features to encode.
Categorical feature encoding complete.


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,54865.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,55054.0,109.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,55055.0,52.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,46236.0,34.0,1.0,1.0,6.0,6.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,54863.0,3.0,2.0,0.0,12.0,0.0,6.0,6.0,6.0,0.0,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [ ]:
# encoded_df['label'].value_counts()  # Check the distribution of the target variable after encoding

label
0     2273097
4      231073
10     158930
2      128027
3       10293
7        7938
11       5897
6        5796
5        5499
1        1966
12       1507
14        652
9          36
13         21
8          11
Name: count, dtype: int64

In [20]:
def apply_min_max_scaling(df, exclude_columns=[]):
    """
    Applies Min-Max scaling to numerical features, excluding specified columns.
    """
    print("Applying Min-Max scaling...")
    numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
    
    # Exclude columns that should not be scaled (e.g., already encoded target)
    cols_to_scale = [col for col in numerical_cols if col not in exclude_columns]

    if not cols_to_scale:
        print("  No numerical columns to scale.")
        return df, None

    scaler = MinMaxScaler()
    df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
    print(f"  Scaled {len(cols_to_scale)} numerical columns.")
    print("Min-Max scaling complete.")
    return df, scaler # Return scaler for potential inverse transform/feature scaling consistency


In [21]:
scaled_df, scaler = apply_min_max_scaling(encoded_df, exclude_columns=['label'])  # Exclude target column from scaling
scaled_df.head()

Applying Min-Max scaling...
  Scaled 78 numerical columns.
Min-Max scaling complete.


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,0.873146,1.389920e-07,0.029651,0.000000,0.00366,0.000000,0.006117,0.074788,0.029528,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.876154,1.059814e-06,0.000000,0.024056,0.00183,0.000096,0.006117,0.074788,0.029528,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.876170,5.646550e-07,0.000000,0.024056,0.00183,0.000096,0.006117,0.074788,0.029528,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.735821,4.082890e-07,0.000000,0.024056,0.00183,0.000096,0.006117,0.074788,0.029528,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.873115,1.389920e-07,0.029651,0.000000,0.00366,0.000000,0.006117,0.074788,0.029528,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [22]:
def preprocess_dataset(df, dataset_name, target_col):
    """
    Orchestrates the preprocessing steps.
    """
    print(f"\n--- Starting preprocessing for {dataset_name} ---")
    
    # 1. Handle Missing Values
    df = handle_missing_values(df)
    
    # 2. Detect and Handle Outliers
    df = detect_and_handle_outliers_zscore(df)
    
    # 3. Encode Categorical Features
    # Make a copy of the target column before encoding the rest to avoid issues
    df_labels = df[target_col].copy()
    df = encode_categorical_features(df, target_column=target_col)
    
    # Re-attach the numerical labels if they were separated (only if target was originally object)
    if df_labels.dtype == 'object': # Check if the original target was categorical
         df[target_col] = target_label_encoder.transform(df_labels) # Use the global encoder

    # Ensure the target column is at the end or handle it specifically for scaling if it's numerical and shouldn't be scaled.
    # For anomaly detection, 'Label' or 'Attack' is often binary (0 for normal, 1 for anomaly), which typically isn't scaled.
    columns_to_exclude_from_scaling = [target_col]
    
    # 4. Apply Min-Max Scaling
    df, scaler = apply_min_max_scaling(df, exclude_columns=columns_to_exclude_from_scaling)
    
    # Reorder columns to place target at the end for consistency if desired
    if target_col in df.columns:
        cols = [col for col in df.columns if col != target_col] + [target_col]
        df = df[cols]

    print(f"--- Preprocessing for {dataset_name} complete. Final shape: {df.shape} ---")
    return df

In [23]:
final_df = preprocess_dataset(all_data_df, dataset_name='CIC-IDS2017', target_col='label')
final_df.head()


--- Starting preprocessing for CIC-IDS2017 ---
Handling missing values...
Missing value handling complete.
Detecting and handling outliers using Z-score (threshold=3)...


C:\Users\vicky\AppData\Local\Temp\ipykernel_33352\1346073590.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[8.79367633 8.79367633 8.79367633 ... 8.79367633 8.79367633 8.79367633]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[z_scores > threshold, col] = np.sign(df[col]) * (mean_val + threshold * std_dev) # Cap to 3 std dev from mean


Outlier handling complete.
Encoding categorical features...
  No other categorical features to encode.
Categorical feature encoding complete.
Applying Min-Max scaling...
  Scaled 78 numerical columns.
Min-Max scaling complete.
--- Preprocessing for CIC-IDS2017 complete. Final shape: (2830743, 79) ---


,destination_port,flow_duration,total_fwd_packets,total_backward_packets,total_length_of_fwd_packets,total_length_of_bwd_packets,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,min_seg_size_forward,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,0.873151,1.390697e-07,0.04364,0.000000,0.005192,0.000000,0.006983,0.075301,0.0346,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.876159,1.060407e-06,0.00000,0.039654,0.002596,0.000212,0.006983,0.075301,0.0346,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.876175,5.649708e-07,0.00000,0.039654,0.002596,0.000212,0.006983,0.075301,0.0346,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.735825,4.085173e-07,0.00000,0.039654,0.002596,0.000212,0.006983,0.075301,0.0346,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.873120,1.390697e-07,0.04364,0.000000,0.005192,0.000000,0.006983,0.075301,0.0346,0.0,...,0.996523,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
